# **SN_AI – Predicting Serie A Match Results Using RNNs**

---

## Introduction / Context
The **SN_AI** project aims to predict Serie A match results from the 2015/2016 season to the ongoing 2025/2026 season.  
We will use **Recurrent Neural Networks (RNNs)** to capture temporal dependencies in historical team data and generate probabilistic match outcome predictions.

**Note:** Match results depend on more than historical results. Real-life factors such as player fitness, absences, strategies, and transfer market dynamics cannot currently be modeled due to insufficient data.

---

## Dataset and Split
- **Dataset:** each match is represented as a JSON object containing date, teams, score, and match number.  

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

**The dataset is divided in this way:**
- 70% for training
- 15% for validation
- 15% for testing

## Features
Main features used for the RNN:

- **Teams:** `home_team_name`, `away_team_name`  
- **Recent performance (last 5 matches):**  
  `home_last5_points`, `away_last5_points`  
  `home_last5_goals`, `away_last5_goals`  
- **Season averages:**  
  `home_avg_goals_scored`, `away_avg_goals_scored`  
  `home_avg_goals_conceded`, `away_avg_goals_conceded`  
- **Elo ratings:**  
  `elo_home_team`, `elo_away_team`, `elo_diff`  
- **Recent performance differences:**  
  `goal_diff_last5`, `points_diff_last5`  
- **Match history last 2 seasons:**  
  `last2yrs_match_history`  

---

## RNN Output
The network should output a Python object:

```python
result_predicted = {
    "home_win_prob": ...,
    "away_win_prob": ...,
    "draw_prob": ...,
    "home_goal_prediction": ...,
    "away_goal_prediction": ...,
    "over_2_5_prob": ...,
    "under_2_5_prob": ...
}
```

## Architecture and Method
- Network type: **RNN** (to capture temporal dependencies)  
- Activation function: **ReLU**  
- Loss function: **cross-entropy** (for result probabilities) and **MSE** (for goals)  
- Backpropagation: **BPTT (Backpropagation Through Time)**  
- Optimizer: **Adam**  
- Regularization: **dropout and L2 weight decay**  
- Validation: train/validation/test as indicated

---

## Evaluation
- Metrics:
  - Accuracy / F1-score / Precision / Recall (for win/draw/loss probabilities)  
  - MAE / MSE (for predicted goals)  
  - Confusion matrix  
- Feature importance analysis to understand which features most influence predictions

---

In [123]:
from pathlib import Path

def read_file(file_path: Path) -> str:
    """Read and return the content of a file."""
    print(f"Reading file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def read_folder_files(folder_path: str) -> dict:
    """Read all files in a folder and return a dict {filename: content}."""
    folder = Path(folder_path)

    return {
        file.name: read_file(file)
        for file in folder.iterdir()
        if file.is_file()
    }

############################################################################################################################################################

files_path= "./json_files"
file_contents = read_folder_files(files_path)

Reading file: json_files/SerieA_0405.json
Reading file: json_files/SerieA_9798.json
Reading file: json_files/SerieA_0203.json
Reading file: json_files/SerieA_1516.json
Reading file: json_files/SerieA_1920.json
Reading file: json_files/SerieA_2526.json
Reading file: json_files/SerieA_0809.json
Reading file: json_files/SerieA_9900.json
Reading file: json_files/SerieA_1415.json
Reading file: json_files/SerieA_0506.json
Reading file: json_files/SerieA_0910.json
Reading file: json_files/SerieA_1213.json
Reading file: json_files/SerieA_2223.json
Reading file: json_files/SerieA_1819.json
Reading file: json_files/SerieA_2425.json
Reading file: json_files/SerieA_9697.json
Reading file: json_files/SerieA_0708.json
Reading file: json_files/SerieA_2324.json
Reading file: json_files/SerieA_9495.json
Reading file: json_files/SerieA_2021.json
Reading file: json_files/SerieA_9899.json
Reading file: json_files/SerieA_1011.json
Reading file: json_files/SerieA_1314.json
Reading file: json_files/SerieA_01

In [124]:
import json
from collections import defaultdict
from datetime import datetime
from enum import IntEnum


def create_team_enum(teams_dict):
    enum_dict = {
        team.upper().replace(" ", "_"): i
        for i, team in enumerate(teams_dict.keys())
    }
    return IntEnum("SerieATeams", enum_dict)


def attach_enum_to_teams(teams_dict, enum_cls):
    return {
        team: (enum_cls[team.upper().replace(" ", "_")], points)
        for team, points in teams_dict.items()
    }


def clean_match(match: dict) -> dict:
    """Remove unnecessary keys from match."""
    keys_to_remove = {"Location", "Group"}
    return {k: v for k, v in match.items() if k not in keys_to_remove}


def update_team_points(match: dict, teams: dict):
    """Update cumulative team points based on match result."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    teams.setdefault(home, 0)
    teams.setdefault(away, 0)

    if home_score > away_score:
        teams[home] += 3
    elif home_score < away_score:
        teams[away] += 3
    else:
        teams[home] += 1
        teams[away] += 1


def track_past_match(match: dict, past_matches: dict, season_number: str):
    """
    Store past match results using a tuple key: (home_team, away_team).
    Save numeric goals and timestamp in milliseconds.
    """
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    key = (home, away)
    match_day_str = match["DateUtc"]  # es: "2015-08-22 18:00:00Z"

    match_dt = datetime.strptime(match_day_str, "%Y-%m-%d %H:%M:%SZ")
    match_day_m = int(match_dt.timestamp() * 1000)

    past_matches.setdefault(key, [])
    past_matches[key].append({
        "home_goals": home_score,
        "away_goals": away_score,
        "matchDay": match_day_m,
        "season": season_number
    })


def parse_json(json_content: str, teams: dict, past_matches: dict, season_number: str):
    """Parse JSON content and update statistics."""
    try:
        matches = json.loads(json_content)

        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        cleaned_matches = []

        for match in matches:
            match = clean_match(match)

            if match["HomeTeamScore"] is not None and match["AwayTeamScore"] is not None:
                update_team_points(match, teams)
                track_past_match(match, past_matches, season_number)

            cleaned_matches.append(match)

        return cleaned_matches

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []


def parse_dataset(file_contents: dict, teams: dict, past_matches: dict):
    """Parse all files in a dataset."""
    parsed_contents = {}

    for file, content in file_contents.items():
        season_number = file.split("_")[1].split(".")[0]
        parsed_contents[file] = parse_json(content, teams, past_matches, season_number)

    return parsed_contents

#####################################################################
# STRUCTURES
#####################################################################

all_possible_past_matches = {}
all_possible_teams = {}

train_parsed_contents = parse_dataset(
    file_contents, all_possible_teams, all_possible_past_matches
)

SerieATeams = create_team_enum(all_possible_teams)
num_teams = len(SerieATeams)

all_possible_teams = attach_enum_to_teams(all_possible_teams, SerieATeams)

#####################################################################
# SORT PAST MATCHES BY matchDay
#####################################################################

for match_key, match_list in all_possible_past_matches.items():
    match_list.sort(key=lambda x: x["matchDay"])

for match_key, match_list in all_possible_past_matches.items():
    match_days = [match["matchDay"] for match in match_list]
    if match_days != sorted(match_days):
        print(f"Error: Match list for {match_key} is not sorted by matchDay.")

#####################################################################
# BUILD team_matches
#####################################################################

team_matches = {team: [] for team in all_possible_teams.keys()}

for match_key, match_list in all_possible_past_matches.items():
    home_team, away_team = match_key  # <-- qui non c'è più split("-")

    for match in match_list:
        # record dal punto di vista della squadra di casa
        team_matches[home_team].append({
            "home_team": home_team,
            "away_team": away_team,
            "is_home": True,
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        })

        # record dal punto di vista della squadra in trasferta
        team_matches[away_team].append({
            "home_team": home_team,
            "away_team": away_team,
            "is_home": False,
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        })

#####################################################################
# SORT team_matches BY matchDay
#####################################################################

for team, matches in team_matches.items():
    matches.sort(key=lambda x: x["matchDay"])

for team, matches in team_matches.items():
    match_days = [match["matchDay"] for match in matches]
    if match_days != sorted(match_days):
        print(f"Error: Match list for team {team} is not sorted by matchDay.")

# divide all_possible_past_matches in train, eval and test based on matchDay and season and creating dictionaries, train 70%, eval 15% and test 15%
train_data = {}
eval_data = {}
test_data = {}

train_seasons = ["9495", "9596", "9697", "9798", "9899", "9900", "0001", "0102", "0203", "0304", "0405", "0506", "0607", "0708", "0809", "0910", "1011", "1112", "1213", "1314", "1415", "1516"]
eval_seasons = ["1617", "1718", "1819", "1920", "2021"]
test_seasons = ["2122", "2223", "2324", "2425", "2526"]

for match_key, matches in all_possible_past_matches.items():
    for match in matches:
        season = match["season"]
        if season in train_seasons:
            train_data.setdefault(match_key, []).append(match)
        elif season in eval_seasons:
            eval_data.setdefault(match_key, []).append(match)
        elif season in test_seasons:
            test_data.setdefault(match_key, []).append(match)

train_matches = {match_key: matches for match_key, matches in train_data.items() if matches}
eval_matches = {match_key: matches for match_key, matches in eval_data.items() if matches}
test_matches = {match_key: matches for match_key, matches in test_data.items() if matches}

season_order = ["9495", "9596", "9697", "9798", "9899", "9900", "0001", "0102", "0203", "0304", "0405", "0506", "0607", "0708", "0809", "0910", "1011", "1112", "1213", "1314", "1415", "1516", "1617", "1718", "1819", "1920", "2021", "2122", "2223", "2324", "2425", "2526"]
# group train_matches, eval_matches and test_matches by season
train_matches_by_season = defaultdict(list)
for match_key, matches in train_matches.items():
    for match in matches:
        m = {
            "home_team": match_key[0],
            "away_team": match_key[1],
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        }
        train_matches_by_season[match["season"]].append(m)

for season, matches in train_matches_by_season.items():
    matches.sort(key=lambda x: x["matchDay"])
    match_days = [match["matchDay"] for match in matches]
    if match_days != sorted(match_days):
        print(f"Error: Match list for season {season} in train_matches is not sorted by matchDay.")

# sort train_matches_by_season by season_order
train_matches_by_season = dict(sorted(train_matches_by_season.items(), key=lambda x: season_order.index(x[0])))

eval_matches_by_season = defaultdict(list)
for match_key, matches in eval_matches.items():
    for match in matches:
        m = {
            "home_team": match_key[0],
            "away_team": match_key[1],
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        }
        eval_matches_by_season[match["season"]].append(m)

for season, matches in eval_matches_by_season.items():
    matches.sort(key=lambda x: x["matchDay"])
    match_days = [match["matchDay"] for match in matches]
    if match_days != sorted(match_days):
        print(f"Error: Match list for season {season} in eval_matches is not sorted by matchDay.")

# sort eval_matches_by_season by season_order
eval_matches_by_season = dict(sorted(eval_matches_by_season.items(), key=lambda x: season_order.index(x[0])))

test_matches_by_season = defaultdict(list)
for match_key, matches in test_matches.items():
    for match in matches:
        m = {
            "home_team": match_key[0],
            "away_team": match_key[1],
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        }
        test_matches_by_season[match["season"]].append(m)

for season, matches in test_matches_by_season.items():
    matches.sort(key=lambda x: x["matchDay"])
    match_days = [match["matchDay"] for match in matches]
    if match_days != sorted(match_days):
        print(f"Error: Match list for season {season} in test_matches is not sorted by matchDay.")
    
# sort test_matches_by_season by season_order
test_matches_by_season = dict(sorted(test_matches_by_season.items(), key=lambda x: season_order.index(x[0])))

In [ ]:
def build_teams_in_season(matches):
    ret = {}
    for season, matches in matches.items():
        ret[season] = set()  # uso set per evitare duplicati
    
    for match in matches:
        ret[season].add(match["home_team"])
        ret[season].add(match["away_team"])
    
    return ret

# =========================================================
# FEATURE ENGINEERING
# =========================================================
def find_previous5_team_matches(team, matchday, team_matches):
    matches = team_matches.get(team, [])
    previous_matches = [m for m in matches if m["matchDay"] < matchday]
    return previous_matches[-5:]


def previous5_points(team, previous5_matches):
    points = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            if home_score > away_score:
                points += 3
            elif home_score == away_score:
                points += 1
        elif team == away_team:
            if away_score > home_score:
                points += 3
            elif away_score == home_score:
                points += 1
    return points


def previous5_goals(team, previous5_matches):
    goals = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            goals += int(home_score)
        elif team == away_team:
            goals += int(away_score)
    return goals


def previous5_goals_conceded(team, previous5_matches):
    conceded = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            conceded += int(away_score)
        elif team == away_team:
            conceded += int(home_score)
    return conceded


def previous5_goal_diff(team, previous5_matches):
    goal_diff = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            goal_diff += int(home_score) - int(away_score)
        elif team == away_team:
            goal_diff += int(away_score) - int(home_score)

    return goal_diff


def previous5_points_mean(team, previous5_matches):
    return round(previous5_points(team, previous5_matches) / len(previous5_matches), 2) if len(previous5_matches) > 0 else 0.0


def previous5_win_draw_loss_rates(team, previous5_matches):
    wins = draws = losses = 0
    total = len(previous5_matches)

    if total == 0:
        return 0.0, 0.0, 0.0

    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            if home_score > away_score:
                wins += 1
            elif home_score == away_score:
                draws += 1
            else:
                losses += 1
        elif team == away_team:
            if away_score > home_score:
                wins += 1
            elif away_score == home_score:
                draws += 1
            else:
                losses += 1

    return wins / total, draws / total, losses / total


def season_stats_until_matchday(team, matchday, season, team_matches):
    goals_scored = 0
    goals_conceded = 0
    points = 0
    matches_count = 0

    matches = team_matches.get(team, [])

    for match in matches:
        if match["season"] == season and match["matchDay"] < matchday:
            home_team = match["home_team"]
            away_team = match["away_team"]
            home_score = int(match["home_goals"])
            away_score = int(match["away_goals"])

            if team == home_team:
                goals_scored += home_score
                goals_conceded += away_score
                matches_count += 1
                if home_score > away_score:
                    points += 3
                elif home_score == away_score:
                    points += 1

            elif team == away_team:
                goals_scored += away_score
                goals_conceded += home_score
                matches_count += 1
                if away_score > home_score:
                    points += 3
                elif away_score == home_score:
                    points += 1

    if matches_count == 0:
        return {
            "avg_scored": 0.0,
            "avg_conceded": 0.0,
            "ppg": 0.0,
            "goal_diff_pg": 0.0
        }

    return {
        "avg_scored": round(goals_scored / matches_count, 3),
        "avg_conceded": round(goals_conceded / matches_count, 3),
        "ppg": round(points / matches_count, 3),
        "goal_diff_pg": round((goals_scored - goals_conceded) / matches_count, 3)
    }

def create_label(home_goals, away_goals):
    if home_goals > away_goals:
        return 0   # home win
    elif home_goals == away_goals:
        return 1   # draw
    else:
        return 2   # away win


# =========================================================
# ELO
# =========================================================
def update_elo(home_rating, away_rating, home_goals, away_goals, k=20, home_advantage=80):
    # actual result
    if home_goals > away_goals:
        s_home = 1.0
        s_away = 0.0
    elif home_goals < away_goals:
        s_home = 0.0
        s_away = 1.0
    else:
        s_home = 0.5
        s_away = 0.5

    # expected result
    adjusted_home = home_rating + home_advantage
    e_home = 1.0 / (1.0 + 10 ** ((away_rating - adjusted_home) / 400.0))
    e_away = 1.0 - e_home

    # margin multiplier
    goal_diff = abs(home_goals - away_goals)
    if goal_diff <= 1:
        margin_multiplier = 1.0
    elif goal_diff == 2:
        margin_multiplier = 1.5
    else:
        margin_multiplier = 1.75 + 0.1 * (goal_diff - 3)

    new_home = home_rating + k * margin_multiplier * (s_home - e_home)
    new_away = away_rating + k * margin_multiplier * (s_away - e_away)

    return new_home, new_away

# =========================================================
# BUILD FEATURES
# =========================================================
def build_features_and_labels(
    matches_dict,
    all_possible_teams,
    team_matches_map,
    current_elo_ratings=None
):
    if current_elo_ratings is None:
        current_elo_ratings = {team: 1500.0 for team in all_possible_teams.keys()}
    else:
        current_elo_ratings = current_elo_ratings.copy()

    all_matches_flat = []
    for season, results in matches_dict.items():
        for res in results:
            all_matches_flat.append({
                "season": season,
                **res
            })

    all_matches_flat.sort(key=lambda x: x["matchDay"])

    features_teams_ids_list = []
    features_list = []
    labels_list = []

    for m in all_matches_flat:
        h_team = m["home_team"]
        a_team = m["away_team"]
        m_day = m["matchDay"]
        season_num = m["season"]

        # Elo pre-match
        elo_h_before = current_elo_ratings[h_team]
        elo_a_before = current_elo_ratings[a_team]
        elo_diff = elo_h_before - elo_a_before

        # Ultime 5 partite precedenti
        h_prev = find_previous5_team_matches(h_team, m_day, team_matches_map)
        a_prev = find_previous5_team_matches(a_team, m_day, team_matches_map)

        h_prev_points = previous5_points(h_team, h_prev)
        a_prev_points = previous5_points(a_team, a_prev)

        h_prev_goal_diff = previous5_goal_diff(h_team, h_prev)
        a_prev_goal_diff = previous5_goal_diff(a_team, a_prev)

        h_win_rate, _, _ = previous5_win_draw_loss_rates(h_team, h_prev)
        a_win_rate, _, _ = previous5_win_draw_loss_rates(a_team, a_prev)

        # Statistiche stagionali fino al match
        h_season = season_stats_until_matchday(h_team, m_day, season_num, team_matches_map)
        a_season = season_stats_until_matchday(a_team, m_day, season_num, team_matches_map)

        # Differenze finali
        prev_points_diff = h_prev_points - a_prev_points
        prev_goal_diff_diff = h_prev_goal_diff - a_prev_goal_diff
        season_goal_diff_pg_diff = h_season["goal_diff_pg"] - a_season["goal_diff_pg"]

        features = [
            elo_diff,
            prev_points_diff,
            prev_goal_diff_diff,
            season_goal_diff_pg_diff,
            abs(elo_diff),
            abs(prev_points_diff),
            h_win_rate - a_win_rate
        ]

        label = create_label(m["home_goals"], m["away_goals"])

        features_teams_ids_list.append([
            all_possible_teams[h_team][0],
            all_possible_teams[a_team][0]
        ])
        features_list.append(features)
        labels_list.append(label)

        # Aggiorna Elo dopo il match
        new_h, new_a = update_elo(
            elo_h_before,
            elo_a_before,
            m["home_goals"],
            m["away_goals"]
        )
        current_elo_ratings[h_team] = new_h
        current_elo_ratings[a_team] = new_a

    return (
        features_teams_ids_list,
        features_list,
        labels_list,
        current_elo_ratings,
    )

def build_features_per_season(previous_season_num, actual_season_num, matches_df, teams_df, seasons):
    actual_season = season[actual_season_num]

    if previous_season is not None:
        previous_season = seasons[previous_season_num]
        season_matches_previous_season = matches_df[previous_season]
        teams_previous_season = teams_df[previous_season]
    season_matches = matches_df[actual_season]
    teams_actual_season = teams_df[actual_season]

    return {"ciao": 1234, "denis": 2, "bruno": 2, "suca": 3}
# =========================================================
# DATA PREPARATION
# =========================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

initial_elo = {team: 1500.0 for team in all_possible_teams.keys()}

train_season_teams = build_teams_in_season(train_matches_by_season)
eval_season_teams = build_teams_in_season(eval_matches_by_season)
test_season_teams = build_teams_in_season(test_matches_by_season)

season_features = {}


# TRAIN
train_team_ids, X_train_num, Y_train, elo_after_train = build_features_and_labels(
    train_matches_by_season,
    all_possible_teams,
    team_matches,
    current_elo_ratings=initial_elo
)

# EVAL
eval_team_ids, X_eval_num, Y_eval, elo_after_eval= build_features_and_labels(
    eval_matches_by_season,
    all_possible_teams,
    team_matches,
    current_elo_ratings=elo_after_train
)

# TEST
test_team_ids, X_test_num, Y_test, elo_after_test = build_features_and_labels(
    test_matches_by_season,
    all_possible_teams,
    team_matches,
    current_elo_ratings=elo_after_eval
)



X_train_num = scaler.fit_transform(X_train_num)
X_eval_num = scaler.transform(X_eval_num)
X_test_num = scaler.transform(X_test_num)

X_train = [list(train_team_ids[i]) + list(X_train_num[i]) for i in range(len(X_train_num))]
X_eval = [list(eval_team_ids[i]) + list(X_eval_num[i]) for i in range(len(X_eval_num))]
X_test = [list(test_team_ids[i]) + list(X_test_num[i]) for i in range(len(X_test_num))]

features_num = len(X_train[0])
print("Example feature vector:", X_train[0])
print("Total features including team ids:", features_num)


TypeError: string indices must be integers, not 'str'

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MatchOutcomePredictorMLP(nn.Module):
    def __init__(self, num_teams, num_features, num_classes=3, embedding_size=48):
        super().__init__()

        self.team_embedding = nn.Embedding(num_teams, embedding_size)

        numeric_features_size = num_features - 2  # escludi home_team_id e away_team_id

        # home_emb + away_emb + emb_diff + emb_prod + numeric features
        input_size = numeric_features_size + (4 * embedding_size)

        self.mlp = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        home_team_ids = x[:, 0].long()
        away_team_ids = x[:, 1].long()
        numeric_features = x[:, 2:]

        home_emb = self.team_embedding(home_team_ids)
        away_emb = self.team_embedding(away_team_ids)

        emb_diff = home_emb - away_emb
        emb_prod = home_emb * away_emb

        x = torch.cat(
            [home_emb, away_emb, emb_diff, emb_prod, numeric_features],
            dim=1
        )

        return self.mlp(x)

# Define a custom Dataset
class MatchDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


batch_size = 32

train_dataset = MatchDataset(X_train, Y_train)
eval_dataset = MatchDataset(X_eval, Y_eval)
test_dataset = MatchDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MatchOutcomePredictorMLP(
    num_teams=num_teams,
    num_features=features_num,
    num_classes=3,
    embedding_size=16
).to(device)

# give class weights 
classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight={0: 1.0, 1: 1.8, 2: 1.3}, classes=classes, y=Y_train)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    threshold=0.002,
    threshold_mode="abs",
    cooldown=1,
    min_lr=1e-5
)

Class weights: tensor([1.0000, 1.8000, 1.3000])


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import copy
import numpy as np

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    return running_loss / len(dataloader.dataset)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * X_batch.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)

    return epoch_loss, acc, np.array(all_preds), np.array(all_targets), np.array(all_probs)


num_epochs = 80
patience = 10
best_eval_loss = float("inf")
best_state_dict = None
epochs_without_improvement = 0

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    eval_loss, eval_acc, eval_preds, eval_targets, eval_probs = evaluate(model, eval_loader, criterion, device)

    scheduler.step(eval_loss)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Eval Loss: {eval_loss:.4f} | "
        f"Eval Accuracy: {eval_acc:.4f}"
    )

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state_dict)

test_loss, test_acc, test_preds, test_targets, test_probs = evaluate(model, test_loader, criterion, device)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

print(classification_report(
    test_targets,
    test_preds,
    target_names=["Home Win", "Draw", "Away Win"]
))

print("Confusion matrix:")
print(confusion_matrix(test_targets, test_preds))

Epoch 1/80 | Train Loss: 1.0889 | Eval Loss: 1.0277 | Eval Accuracy: 0.4882
Epoch 2/80 | Train Loss: 1.0464 | Eval Loss: 1.0248 | Eval Accuracy: 0.5171
Epoch 3/80 | Train Loss: 1.0354 | Eval Loss: 1.0134 | Eval Accuracy: 0.5192
Epoch 4/80 | Train Loss: 1.0275 | Eval Loss: 1.0136 | Eval Accuracy: 0.5003
Epoch 5/80 | Train Loss: 1.0229 | Eval Loss: 1.0120 | Eval Accuracy: 0.5155
Epoch 6/80 | Train Loss: 1.0190 | Eval Loss: 1.0164 | Eval Accuracy: 0.5161
Epoch 7/80 | Train Loss: 1.0177 | Eval Loss: 1.0168 | Eval Accuracy: 0.5061
Epoch 8/80 | Train Loss: 1.0160 | Eval Loss: 1.0126 | Eval Accuracy: 0.5161
Epoch 9/80 | Train Loss: 1.0115 | Eval Loss: 1.0084 | Eval Accuracy: 0.5187
Epoch 10/80 | Train Loss: 1.0101 | Eval Loss: 1.0143 | Eval Accuracy: 0.5055
Epoch 11/80 | Train Loss: 1.0078 | Eval Loss: 1.0151 | Eval Accuracy: 0.5161
Epoch 12/80 | Train Loss: 1.0078 | Eval Loss: 1.0199 | Eval Accuracy: 0.4839
Epoch 13/80 | Train Loss: 1.0078 | Eval Loss: 1.0168 | Eval Accuracy: 0.5092
Epoch 14

In [ ]:
label_names = ["🏠", "🟰", "✈️"]

print("\nPredizioni sui match di test:")
for i in range(len(X_test)):
    home_team_id = int(X_test[i][0])
    away_team_id = int(X_test[i][1])

    home_team = list(all_possible_teams.keys())[home_team_id]
    away_team = list(all_possible_teams.keys())[away_team_id]

    probs = test_probs[i]
    predicted_label = test_preds[i]
    true_label = test_targets[i]
    correct = predicted_label == true_label

    print(f"Match: {home_team} vs {away_team}")
    print(f"Predicted: {label_names[predicted_label]} | True: {label_names[true_label]} | Correct: {'✅' if correct else '❌'}")
    print(f"Probabilities -> Home: {probs[0]:.3f}, Draw: {probs[1]:.3f}, Away: {probs[2]:.3f}\n")


Predizioni sui match di test:
Match: Inter vs Genoa
Predicted: 🏠 | True: 🏠 | Correct: ✅
Probabilities -> Home: 0.613, Draw: 0.281, Away: 0.106

Match: Hellas Verona vs Sassuolo
Predicted: 🟰 | True: ✈️ | Correct: ❌
Probabilities -> Home: 0.314, Draw: 0.408, Away: 0.278

Match: Empoli vs Lazio
Predicted: ✈️ | True: ✈️ | Correct: ✅
Probabilities -> Home: 0.255, Draw: 0.295, Away: 0.450

Match: Torino vs Atalanta
Predicted: ✈️ | True: ✈️ | Correct: ✅
Probabilities -> Home: 0.190, Draw: 0.333, Away: 0.477

Match: Udinese vs Juventus
Predicted: ✈️ | True: 🟰 | Correct: ❌
Probabilities -> Home: 0.207, Draw: 0.331, Away: 0.462

Match: Bologna vs Salernitana
Predicted: 🟰 | True: 🏠 | Correct: ❌
Probabilities -> Home: 0.387, Draw: 0.387, Away: 0.226

Match: Roma vs Fiorentina
Predicted: 🏠 | True: 🏠 | Correct: ✅
Probabilities -> Home: 0.496, Draw: 0.348, Away: 0.156

Match: Napoli vs Venezia
Predicted: 🏠 | True: 🏠 | Correct: ✅
Probabilities -> Home: 0.719, Draw: 0.214, Away: 0.067

Match: Cagliari